# Testing: all four algorithms together
Run after `./kaggle.sh get` has downloaded the 4 training notebooks into `downloads/`. Everything is saved in `results/`.

## 1. Load

In [ ]:
import os, sys

ROOT = os.path.abspath(".." if os.path.basename(os.getcwd()) == "testing" else ".")
os.chdir(ROOT)
sys.path.insert(0, ROOT)
os.environ["MACURA_TASK"] = "race3"
os.environ.setdefault("MUJOCO_GL", "disable")

DOWNLOADS = "downloads"
OUT = "results"
EPISODES = 50           # new test scenarios per model
VIDEOS = True

In [ ]:
import json
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, Video, display
from src.config import config as cfg
from src.analysis import results as R, report, testing as T
from src.viz import figures as F, video as V, test_figures as TF

In [ ]:
FOLDERS = R.find_downloads(DOWNLOADS)
RUNS = R.load_runs(*FOLDERS)
display(Markdown(R.where_table(RUNS)))

## 2. Training results

In [ ]:
display(Markdown(R.summary_markdown(RUNS)))
display(Markdown(R.per_seed_markdown(RUNS)))
for other in ["mbpo", "m2ac", "sac"]:
    display(Markdown(R.pair_table(RUNS, "macura", other)))
display(Markdown(R.updates_table(RUNS)))
display(Markdown(R.chute_table(RUNS)))

In [ ]:
REFS = R.autopilot_reference(cfg.ENV, cache=f"{OUT}/autopilot_reference.json")
F.plot_learning_curves(RUNS, REFS, f"{OUT}/plots/learning_curves_smooth.png", window=7); plt.show()
for algo, runs in R.by_algo(RUNS).items():
    F.plot_algo(runs, REFS, f"{OUT}/plots/{algo}_per_seed.png", window=7); plt.show()
F.plot_scores(RUNS, f"{OUT}/plots/scores.png"); plt.show()
F.plot_summary_table(RUNS, f"{OUT}/plots/summary_table.png"); plt.show()

In [ ]:
print(R.trust_summary(RUNS))
made = report.build(RUNS, cfg.CFG, OUT, check=True)
for name in ["model_trust", "kappa", "untrusted_data", "model_check", "crash_by_payload"]:
    if name in made:
        display(Image(made[name]))

## 3. Test on 50 new scenarios

In [ ]:
TESTS = T.test_all(RUNS, cfg.CFG, episodes=EPISODES)
tables = T.test_tables(TESTS)
display(Markdown(tables["summary"]))
display(Markdown(tables["per_seed"]))
for table in tables["paired"]:
    display(Markdown(table))
R.autopilot_reference(cfg.ENV, seed_base=T.TEST_SEED_BASE, episodes=EPISODES, cache=f"{OUT}/autopilot_new_test.json")

In [ ]:
TF.plot_test_scores(TESTS, f"{OUT}/plots/test_scores.png"); plt.show()
TF.plot_test_seeds(TESTS, f"{OUT}/plots/test_seeds.png"); plt.show()
TF.plot_test_conditions(TESTS, f"{OUT}/plots/test_conditions.png"); plt.show()
TF.plot_test_flights(TESTS, cfg.CFG, trace=0, save_path=f"{OUT}/plots/test_flight_0.png"); plt.show()
TF.plot_test_flights(TESTS, cfg.CFG, trace=1, save_path=f"{OUT}/plots/test_flight_1.png"); plt.show()
TF.plot_test_noise(TESTS, save_path=f"{OUT}/plots/test_wind.png"); plt.show()
TF.plot_test_trust(TESTS, f"{OUT}/plots/test_trust.png"); plt.show()

## 4. Test videos

In [ ]:
def trust_of(run):
    return {"ensemble": R.ensemble_checkpoint(run), "kappa": R.kappa_at(run)}

def show(path):
    if path:
        display(Video(path, embed=True, width=960))

if VIDEOS:
    best = {R.NAMES[a]: R.best_run(runs) for a, runs in R.by_algo(RUNS).items()}
    trust = {name: trust_of(r) for name, r in best.items() if r["algo"] == "macura"}
    pilots = {name: R.checkpoint(r) for name, r in best.items()}
    scenarios = (T.TEST_SEED_BASE, T.TEST_SEED_BASE + 1, T.TEST_SEED_BASE + 2)
    show(V.record_race(cfg.CFG, pilots, f"{OUT}/videos/test_all.mp4", seeds=scenarios, trust=trust, frame_step=3, fps=17))

    for algo, runs in R.by_algo(RUNS).items():
        pilots = {f"{R.NAMES[algo]} seed {r['seed']}": R.checkpoint(r) for r in runs}
        seed_trust = {f"{R.NAMES[algo]} seed {r['seed']}": trust_of(r) for r in runs if algo == "macura"}
        show(V.record_race(cfg.CFG, pilots, f"{OUT}/videos/test_{algo}_seeds.mp4", seeds=(T.TEST_SEED_BASE + 3,),
                           trust=seed_trust, frame_step=3, fps=17))

    duo = {name: R.checkpoint(r) for name, r in best.items() if name in ("MACURA", "MBPO")}
    show(V.record_race(cfg.CFG, duo, f"{OUT}/videos/test_30s.mp4", seeds=(T.TEST_SEED_BASE + 4,), trust=trust,
                       frame_step=3, fps=17, max_steps=1500))

## 5. Final comparison

In [ ]:
train = R.summary(RUNS)
test = {}
for result in TESTS:
    test.setdefault(result["algo"], []).append(T.seed_scores(result))

rows = ["| algorithm | avg return while learning | drones broken | test return | test lap progress | test crash rate |",
        "|---|---|---|---|---|---|"]
for algo in test:
    t = test[algo]
    rows.append(f"| {R.NAMES[algo]} | {train[algo]['avg_return'][0]:.0f} | {train[algo]['broken'][0]:.0f} | "
                f"{R.iqm([s['return'] for s in t]):.0f} | {R.iqm([s['lap'] for s in t]):.0%} | "
                f"{R.iqm([s['crash'] for s in t]):.0%} |")
final_table = "\n".join(rows)
display(Markdown(final_table))

summary = [final_table, R.summary_markdown(RUNS), R.per_seed_markdown(RUNS), tables["summary"], tables["per_seed"],
           *tables["paired"], R.where_table(RUNS)]
open(f"{OUT}/summary.md", "w").write("\n\n".join(summary))
json.dump([{"algo": r["algo"], "seed": r["seed"], "episodes": r["episodes"]} for r in TESTS],
          open(f"{OUT}/test_results.json", "w"), default=float)

## 6. Simulator

In [ ]:
from src.envs import drone_env
from src.algorithms.sac import load_agent

env, obs_dim, act_dim = drone_env.make_env(cfg.ENV)
for r in RUNS:
    load_agent(R.checkpoint(r), obs_dim, act_dim, cfg.SAC)
print(f"all {len(RUNS)} models load. In a terminal:")
print("  mjpython simulate.py")
print("  mjpython simulate.py --scenarios 3000 3001 3002")
print("  mjpython simulate.py --algos macura mbpo --seconds 30 --follow MACURA")